In [46]:
import io, requests, pandas as pd

df = pd.read_html("https://www.iposcoop.com/ipos-recently-filed/")

In [47]:
# Filter rows where "Expected To Trade" is "Withdrawn"
df = df[0][df[0]["Expected To Trade"] == "Withdrawn"].copy()
print(len(df))

34


In [48]:
import numpy as np

name = df["Company"].fillna("")

# Order matters: np.select assigns the FIRST condition that matches
conditions = [
    name.str.contains(r"\bTechnologies\b", case=False, regex=True),
    name.str.contains(
        r"\bAcquisition Corp(?:oration)?\b|\bCorp\b", case=False, regex=True
    ),
    name.str.contains(r"\bInc\b|\bIncorporated\b", case=False, regex=True),
    name.str.contains(r"\bGroup\b", case=False, regex=True),
    name.str.contains(r"\bLtd\b|\bLimited\b", case=False, regex=True),
    name.str.contains(r"\bHoldings?\b", case=False, regex=True),
]
choices = ["Technologies", "Acquisition Corp", "Inc.", "Group", "Limited", "Holdings"]

df["Company Type"] = np.select(conditions, choices, default="Other")

df["Company Type"].value_counts()

Company Type
Limited             8
Other               8
Group               5
Acquisition Corp    5
Holdings            4
Technologies        3
Inc.                1
Name: count, dtype: int64

In [49]:
import numpy as np
import pandas as pd
import yfinance as yf


def to_number(value):
    """'$8.00' -> 8.0 ; '-', '', None -> NaN"""
    return float(
        pd.to_numeric(str(value).replace("$", "").replace(",", ""), errors="coerce")
    )


def extract_numeric_values(numeric_str1, numeric_str2):
    """Returns (low, high, average) for two strings."""
    low, high = to_number(numeric_str1), to_number(numeric_str2)
    if np.isnan(low) and np.isnan(high):
        return low, high, np.nan
    return low, high, np.nanmean([low, high])

In [50]:
df[["Price Low", "Price High", "Avg. price"]] = df.apply(
    lambda r: pd.Series(extract_numeric_values(r["Price Low"], r["Price High"])), axis=1
)

In [51]:
cols = ["Shares (millions)", "Est $ Vol (millions)"]

for col in cols:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.replace(r"[$,]", "", regex=True).str.strip(),
        errors="coerce",
    )

df[cols].dtypes  # both should now be float64

Shares (millions)       float64
Est $ Vol (millions)    float64
dtype: object

In [52]:
raw = df[cols].copy()  # run BEFORE the conversion loop

# after converting:
for col in cols:
    lost = raw.loc[df[col].isna() & raw[col].notna(), col].unique()
    print(col, "->", lost)

Shares (millions) -> []
Est $ Vol (millions) -> []


In [53]:
df["Shares_offered_value"] = (df["Shares (millions)"] * df["Avg. price"]).fillna(df["Est $ Vol (millions)"])

In [54]:
cols = ["Shares (millions)", "Est $ Vol (millions)"]

for col in cols:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.replace(r"[$,]", "", regex=True).str.strip(),
        errors="coerce",
    )

df[cols].dtypes  # both should now be float64

Shares (millions)       float64
Est $ Vol (millions)    float64
dtype: object

In [55]:
df["Shares_offered_value"] = (df["Shares (millions)"] * df["Avg. price"]).fillna(
    df["Est $ Vol (millions)"]
)

In [56]:
agg = (
    df.groupby("Company Type")["Shares_offered_value"]
    .sum()
    .sort_values(ascending=False)
)
agg

Company Type
Acquisition Corp    499.9850
Inc.                351.0000
Holdings            311.6575
Other               290.4450
Limited             219.2500
Technologies        184.9000
Group                56.5750
Name: Shares_offered_value, dtype: float64

In [57]:
# Question 1 500

In [58]:
resp = requests.get(
    "https://www.iposcoop.com/2025-pricings/", headers={"User-Agent": "Mozilla/5.0"}
)
ipos = pd.read_html(io.StringIO(resp.text))[0]
print(len(ipos))  # homework expects ~231

231


In [59]:
ipos["Offer Date"] = pd.to_datetime(ipos["Offer Date"], format="%m/%d/%Y")
filtered = ipos[
    (ipos["Offer Date"] < "2025-09-01") & (ipos["Return"] != "0.00%")
].copy()
print(len(filtered))  # expect ~148

146


In [60]:
import yfinance as yf

frames = []
for t in filtered["Symbol"]:
    d = yf.download(t, period="max", progress=False)
    if d.empty:
        continue  # delisted / not on Yahoo
    d.columns = d.columns.get_level_values(0)
    d["Ticker"] = t
    frames.append(d)
stocks_df = pd.concat(frames)

/var/folders/3f/nmjtb9_d3kv620bthjqvt0v80000gn/T/ipykernel_3940/3280838954.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  d = yf.download(t, period="max", progress=False)
/var/folders/3f/nmjtb9_d3kv620bthjqvt0v80000gn/T/ipykernel_3940/3280838954.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  d = yf.download(t, period="max", progress=False)
/var/folders/3f/nmjtb9_d3kv620bthjqvt0v80000gn/T/ipykernel_3940/3280838954.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  d = yf.download(t, period="max", progress=False)
/var/folders/3f/nmjtb9_d3kv620bthjqvt0v80000gn/T/ipykernel_3940/3280838954.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  d = yf.download(t, period="max", progress=False)
/var/folders/3f/nmjtb9_d3kv620bthjqvt0v80000gn/T/ipykernel_3940/3280838954.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
 

In [61]:
import numpy as np

g = stocks_df.groupby("Ticker")["Close"]
stocks_df["growth_252d"] = g.transform(lambda s: s / s.shift(252))
stocks_df["volatility"] = g.transform(lambda s: s.rolling(30).std() * np.sqrt(252))

In [62]:
stocks_df["Sharpe"] = (stocks_df["growth_252d"] - 0.05) / stocks_df["volatility"]

In [63]:
snapshot = stocks_df[stocks_df.index == "2026-09-11"]
print(snapshot[["growth_252d", "volatility", "Sharpe"]].describe())

Price  growth_252d  volatility      Sharpe
count   130.000000  131.000000  130.000000
mean      1.057613   27.488300         NaN
std       3.049007   66.890026         NaN
min       0.001005    0.000000        -inf
25%       0.141035    2.265163    0.012042
50%       0.594534    7.885748    0.050142
75%       1.038821   23.452319    0.133619
max      33.638270  627.698106         inf


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/core/_methods.py:49: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


In [ ]:
# 2 answer is 0.04

In [64]:
g = stocks_df.groupby("Ticker")["Close"]
for m in range(1, 13):
    stocks_df[f"future_growth_{m}_m"] = g.transform(lambda s: s.shift(-21 * m) / s)

In [65]:
first_days = (
    stocks_df.reset_index()
    .groupby("Ticker")["Date"]
    .min()
    .reset_index()
    .rename(columns={"Date": "min_date"})
)

In [66]:
all_rows = stocks_df.reset_index()
entry = all_rows.merge(
    first_days, left_on=["Ticker", "Date"], right_on=["Ticker", "min_date"], how="inner"
)
print(len(entry))  # ~134, one per stock

132


In [67]:
cols = [f"future_growth_{m}_m" for m in range(1, 13)]
stats = entry[cols].describe()
print(stats)
print(stats.loc["50%"].idxmax(), stats.loc["50%"].max())

       future_growth_1_m  future_growth_2_m  future_growth_3_m  \
count         132.000000         131.000000         131.000000   
mean            1.119635           1.294861           1.252014   
std             1.254616           1.957999           1.912626   
min             0.072941           0.106197           0.090147   
25%             0.708835           0.584863           0.442477   
50%             0.935351           0.890411           0.827160   
75%             1.098038           1.152849           1.149995   
max             9.466019          16.066783          16.344464   

       future_growth_4_m  future_growth_5_m  future_growth_6_m  \
count         131.000000         131.000000         131.000000   
mean            1.135939           1.084528           1.114789   
std             1.558464           1.468733           1.668273   
min             0.063927           0.032245           0.022075   
25%             0.392759           0.400704           0.308026   
50%      

In [ ]:
# 1 month

In [70]:
%pip install gdown
import gdown


file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=False)
df = pd.read_parquet("data.parquet", engine="pyarrow")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gdown]
Note: you may need to restart the kernel to use updated packages.


Downloading...
From (original): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-
From (redirected): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-&confirm=t&uuid=66ad9aa0-c231-4078-b17a-c65988d7e32e
To: /Users/donnabrown/Desktop/stock-market-analytics/data.parquet
100%|██████████| 130M/130M [00:02<00:00, 45.0MB/s] 


In [71]:
print(df.columns.tolist())
print(df[["Date", "Ticker", "rsi", "growth_future_30d"]].head())

['Open', 'High', 'Low', 'Close_x', 'Volume', 'Dividends', 'Stock Splits', 'Ticker', 'Year', 'Month', 'Weekday', 'Date', 'growth_1d', 'growth_3d', 'growth_7d', 'growth_30d', 'growth_90d', 'growth_365d', 'growth_future_30d', 'SMA10', 'SMA20', 'growing_moving_average', 'high_minus_low_relative', 'volatility', 'is_positive_growth_30d_future', 'ticker_type', 'index_x', 'adx', 'adxr', 'apo', 'aroon_1', 'aroon_2', 'aroonosc', 'bop', 'cci', 'cmo', 'dx', 'macd', 'macdsignal', 'macdhist', 'macd_ext', 'macdsignal_ext', 'macdhist_ext', 'macd_fix', 'macdsignal_fix', 'macdhist_fix', 'mfi', 'minus_di', 'mom', 'plus_di', 'dm', 'ppo', 'roc', 'rocp', 'rocr', 'rocr100', 'rsi', 'slowk', 'slowd', 'fastk', 'fastd', 'fastk_rsi', 'fastd_rsi', 'trix', 'ultosc', 'willr', 'index_y', 'ad', 'adosc', 'obv', 'atr', 'natr', 'ht_dcperiod', 'ht_dcphase', 'ht_phasor_inphase', 'ht_phasor_quadrature', 'ht_sine_sine', 'ht_sine_leadsine', 'ht_trendmod', 'avgprice', 'medprice', 'typprice', 'wclprice', 'index', 'cdl2crows', '

In [72]:
RSI_THRESHOLD = 30

In [73]:
selected_df = df[
    (df["Date"] >= "2000-01-01")
    & (df["Date"] <= "2025-06-01")
    & (df["rsi"] < RSI_THRESHOLD)
].copy()

print(len(selected_df))  # homework says ~5,206 trades

5206


In [74]:
net_income = 1000 * (selected_df["growth_future_30d"] - 1).sum()
print(round(net_income / 1000, 1), "thousand $")

65.8 thousand $


In [75]:
print(selected_df["growth_future_30d"].mean())  # ~1.0126 (average +1.26%)
print((selected_df["growth_future_30d"] > 1).mean())  # ~0.5513 (55.13% win rate)

1.0126403360751977
0.5512869765655013
